# 排序、二分查找与二叉树


本篇对应《Python 进阶》第 7 天《算法详解：二叉树》资料，按课程顺序整理为三部分：排序算法、二分查找，以及树和二叉树。代码保留随堂笔记的函数名和实现方式；课程代码仅用于阅读，不在 Notebook 中执行。


## 一、排序算法的稳定性

排序是按照一个或多个关键字的大小，把记录排列成递增或递减顺序。若待排序记录有相同关键字，排序后这些记录的**相对次序仍保持不变**，称为稳定排序；相对次序发生变化，则是不稳定排序。这里比较的是相同关键字记录之间的相对顺序，不是它们的绝对下标。

![排序稳定性示意图](07-img/01-排序稳定性.jpg)

课程列举的稳定算法有冒泡、插入、归并和基数排序；选择、快速、希尔和堆排序通常归为不稳定算法。本节实现冒泡、选择和插入三种。


## 二、冒泡排序

冒泡排序反复比较相邻元素：升序排列时，若前一个元素大于后一个，就交换它们。一轮结束后，本轮最大值被交换到未排序区间的末端；下一轮缩短比较范围，直到全部有序。

以 5 个元素为例，外层需要 `n - 1` 轮，第 `i` 轮（从 `0` 开始）比较 `n - 1 - i` 次。代码中的 `j` 与 `j + 1` 始终是一对相邻位置。


```python
def bubble_sort(my_list):
    n = len(my_list)
    for i in range(n - 1):
        count = 0
        for j in range(n - 1 - i):
            if my_list[j] > my_list[j + 1]:
                count += 1
                my_list[j], my_list[j + 1] = my_list[j + 1], my_list[j]
        print(f'第 {i + 1} 轮交换了{count}次')
        if count == 0:
            break


if __name__ == '__main__':
    my_list = [3, 2, 5, 7, 6, 6]
    bubble_sort(my_list)
    print(my_list)
```

`bubble_sort` 直接修改传入的列表，没有返回新列表。每轮把 `count` 重新置零；如果一轮没有交换，说明列表已经有序，可以提前 `break`。因此最好时间复杂度为 `O(n)`，最坏为 `O(n²)`，额外空间为 `O(1)`，且相等元素不会交换，所以是稳定排序。

- 外层 `n - 1`：最后一个元素确定后不需要再开一轮。
- 内层 `n - 1 - i`：`j + 1` 不能越界，同时跳过已经确定的后缀。
- `count == 0`：利用有序数据的特征提前结束。


## 三、选择排序

选择排序每轮把未排序区间的第一个位置暂定为最小值，用 `min_index` 记录真正的最小元素。扫描剩余区间后，只在最小下标发生变化时交换一次。外层 `i` 左侧是已经排好序的区间，内层从 `i + 1` 扫到列表末尾。


```python
def select_sort(my_list):
    n = len(my_list)
    for i in range(n - 1):
        min_index = i
        for j in range(i + 1, n):
            if my_list[j] < my_list[min_index]:
                min_index = j
        if min_index != i:
            my_list[min_index], my_list[i] = my_list[i], my_list[min_index]


if __name__ == '__main__':
    my_list = [2, 3, 5, 6, 7]
    select_sort(my_list)
    print(my_list)
```

选择排序无论数据原本是否有序，都要完成近似 `n² / 2` 次比较，因此最好和最坏时间复杂度都是 `O(n²)`，额外空间为 `O(1)`。它通常是不稳定的：例如 `[7前, 3, 4, 7后, 2]` 第一轮把 `2` 与前面的 `7` 交换，两个相同关键字 `7` 的相对次序就反转了。


## 四、插入排序

插入排序把列表分成有序区和无序区，默认第一个元素属于有序区。每轮取无序区最左边的元素，向左与有序区比较；较小时交换，遇到不小于它的元素就可以停止。


```python
def insert_sort(my_list):
    n = len(my_list)
    for i in range(1, n):
        for j in range(i, 0, -1):
            if my_list[j] < my_list[j - 1]:
                my_list[j], my_list[j - 1] = my_list[j - 1], my_list[j]
            else:
                break


if __name__ == '__main__':
    my_list = [5, 3, 6, 7, 2]
    insert_sort(my_list)
    print(my_list)
```

内层循环使用 `range(i, 0, -1)`，所以最多向左比较到下标 `1`，对应的前一个位置是 `0`。比较条件使用严格小于，相等元素不会交换，因而插入排序稳定。最好时间复杂度为 `O(n)`，最坏为 `O(n²)`，额外空间为 `O(1)`。


### 三种排序的对比

| 算法 | 核心动作 | 最好时间 | 最坏时间 | 稳定性 | 额外空间 |
| --- | --- | ---: | ---: | --- | ---: |
| 冒泡 | 相邻比较并交换，已排序后缀逐轮扩大 | `O(n)`（提前结束） | `O(n²)` | 稳定 | `O(1)` |
| 选择 | 每轮寻找最小值，最后交换一次 | `O(n²)` | `O(n²)` | 不稳定 | `O(1)` |
| 插入 | 将无序元素向左插入有序区 | `O(n)` | `O(n²)` | 稳定 | `O(1)` |

三种实现都是**原地修改传入列表**，调用后应直接查看原列表；不要把它们误认为返回排序结果的函数。


## 五、二分查找的前提与思路

二分查找要求数据已经按升序排列，并且能够按下标高效访问。每次取中间元素：目标相等就成功；目标更小就只保留左半段；目标更大就只保留右半段。每次排除约一半候选，因此比较次数为 `O(log n)`。

![二分查找的区间变化](07-img/02-二分查找.jpg)

如果列表无序，或者使用不适合按下标访问的存储结构，直接套用二分查找不能得到正确或高效的结果。


### 1. 递归与循环实现

```python
def binary_search_recursion(my_list, target):
    n = len(my_list)
    if n == 0:
        return False
    mid = n // 2
    if my_list[mid] == target:
        return True
    elif target < my_list[mid]:
        return binary_search_recursion(my_list[:mid], target)
    else:
        return binary_search_recursion(my_list[mid + 1:], target)
    return False


def binary_search(my_list, target):
    start = 0
    end = len(my_list) - 1
    while start <= end:
        mid = (start + end) // 2
        if my_list[mid] == target:
            return True
        elif target < my_list[mid]:
            end = mid - 1
        else:
            start = mid + 1
    return False


if __name__ == '__main__':
    my_list = [2, 3, 9, 13, 23, 31, 55, 77, 99]
    print(binary_search_recursion(my_list, 23))
    print(binary_search_recursion(my_list, 25))
    print('-' * 23)
    print(binary_search(my_list, 23))
    print(binary_search(my_list, 25))
```

递归版本用空列表作为终止条件，把列表切片后继续查找；随堂代码最后的 `return False` 位于前面分支返回之后，属于冗余的兜底语句，保留它不影响理解。循环版本维护 `start` 和 `end`，比较后分别更新为 `mid - 1` 或 `mid + 1`，区间为空时返回 `False`。

测试列表为 `[2, 3, 9, 13, 23, 31, 55, 77, 99]`，查找 `23` 预期为 `True`，查找 `25` 预期为 `False`；这里的预期结果是课程示例，不是本 Notebook 的运行输出。


### 2. 复杂度与切片边界

- 循环实现：最坏时间 `O(log n)`，额外空间 `O(1)`，最好情况命中中间位置时为 `O(1)`。
- 递归调用的栈深度为 `O(log n)`，但随堂实现使用 `my_list[:mid]` 和 `my_list[mid + 1:]`，每次递归都会复制列表；因此该具体 Python 实现最坏实际时间可达到 `O(n)`，额外空间也可达到 `O(n)`。

如果只想保留理论上的 `O(log n)` 时间，应改为传递左右下标，而不是复制切片；本篇保留课程实现并把这个实现细节单独标明。


## 六、树的概念与术语

树是一种非线性的层次结构。非空树只有一个根结点；除根以外的每个结点恰有一个父结点，可以有零个或多个子结点；任意结点与其后代组成子树，兄弟结点的子树互不相交。

![树的层次、根、父子和叶结点](07-img/03-树的概念.jpg)

常用术语：结点的度是它拥有的子结点数，树的度是所有结点度的最大值，度为 0 的结点是叶结点；同一父结点的孩子互为兄弟；根在第 1 层；树的高度（深度）是最大层数；沿父结点向上可得到祖先，沿子结点向下可得到后代。


## 七、树的种类、存储与应用

### 1. 常见种类

- **无序树 / 有序树**：有序树关注同一父结点下子结点的次序，并不等于关键字已经排序。
- **二叉树**：每个结点最多有两个子树，并区分左子树和右子树。
- **完全二叉树**：除最后一层外各层都满，最后一层的结点从左到右连续；满二叉树是它的特例。
- **满二叉树**：每层都达到该层的最大结点数，所有叶结点在同一层。
- **平衡二叉树**：课程以每个结点左右子树高度差不超过 1 为判断条件；AVL 树是满足这一条件的二叉搜索树。
- **二叉搜索树（BST）**：左子树所有关键字小于根，右子树所有关键字大于根，左右子树也递归满足该条件；中序遍历结果有序。
- **哈夫曼树**：通过带权路径长度最小构造编码。
- **B 树**：平衡的多路查找树，结点可以有超过两个孩子。课件中把 B 树写成“自平衡的二叉查找树”并不准确。

![完全、满与非完全二叉树](07-img/04-二叉树的种类.jpg)

### 2. 存储方式

- **顺序存储**：用数组位置表达父子关系，完全二叉树最适合；稀疏树会产生空位，未必节省空间。
- **链式存储**：结点通常保存 `item`、`lchild`、`rchild` 三个域。只保存子链接时不能直接从子结点找到父结点；若需要回溯，应另外保存父指针，或从根结点重新定位。

![二叉树结点的链式存储](07-img/05-二叉树的链式存储.jpg)

树结构常见于 XML/HTML 解析、路由、文件目录、数据库索引以及 AI 搜索和决策树。数据库索引示例把关键字与数据位置关联，再组织成树以减少查找范围；这只是课程中的二叉树示意，MySQL 实际索引不应一概理解为普通二叉树。


## 八、二叉树的性质

以下层数和深度从 1 开始计：

1. 第 `i` 层最多有 `2^(i - 1)` 个结点（`i > 0`）。
2. 深度为 `k` 的二叉树最多有 `2^k - 1` 个结点（`k > 0`）。
3. 对任意非空二叉树，叶结点数 `n0` 与度为 2 的结点数 `n2` 满足 `n0 = n2 + 1`。可由边数 `n - 1 = n1 + 2n2`，以及总数 `n = n0 + n1 + n2` 推出。
4. 含 `n` 个结点的完全二叉树深度为 `floor(log2(n)) + 1`，等价于 `ceil(log2(n + 1))`（`n >= 1`）。课件写成未取整的 `log2(n + 1)`，需要补上取整含义；例如 10 个结点的深度是 4。
5. 完全二叉树从上到下、从左到右按 1 开始编号时，结点 `i` 的左孩子是 `2i`，右孩子是 `2i + 1`（不超过结点总数才存在），父结点是 `i // 2`；`i = 1` 的结点是根。Python 列表下标从 0 开始，使用时不要混淆。


## 九、Node 与 BinaryTree

随堂代码把结点和整棵树分成两个类：`Node` 保存数据以及左右孩子引用，`BinaryTree` 只保存根引用并提供插入、遍历方法。`add` 采用层序寻找第一个空位置，所以依次插入数据会形成尽量紧凑的完全树；它**不是**按大小关系建立 BST。

```python
class Node:
    def __init__(self, item):
        self.item = item
        self.lchild = None
        self.rchild = None


class BinaryTree:
    def __init__(self, node=None):
        self.root = node

    def add(self, item):
        new_node = Node(item)
        if self.root is None:
            self.root = new_node
            return
        queue = []
        queue.append(self.root)
        while True:
            node = queue.pop(0)
            if node.lchild is None:
                node.lchild = new_node
                return
            else:
                queue.append(node.lchild)
            if node.rchild is None:
                node.rchild = new_node
                return
            else:
                queue.append(node.rchild)

    def breadth_travel(self):
        if self.root is None:
            return
        queue = []
        queue.append(self.root)
        while len(queue) != 0:
            node = queue.pop(0)
            print(node.item, end=' ')
            if node.lchild is not None:
                queue.append(node.lchild)
            if node.rchild is not None:
                queue.append(node.rchild)

    def preorder_travel(self, root):
        if root is not None:
            print(root.item, end=' ')
            self.preorder_travel(root.lchild)
            self.preorder_travel(root.rchild)

    def inorder_travel(self, root):
        if root is not None:
            self.inorder_travel(root.lchild)
            print(root.item, end=' ')
            self.inorder_travel(root.rchild)

    def postorder_travel(self, root):
        if root is not None:
            self.postorder_travel(root.lchild)
            self.postorder_travel(root.rchild)
            print(root.item, end=' ')
```


### `add` 的过程与队列

1. 新建 `Node(item)`；若根为空，就把它设为根并返回。
2. 建立队列，先把根结点入队。
3. 每次取出队头：左孩子为空就挂上新结点并返回，否则左孩子入队；再检查右孩子，逻辑相同。
4. 队列保证先检查较高层、较左侧的位置，因此插入顺序形成完全树的形状。

课程用列表的 `append` 和 `pop(0)` 模拟先进先出队列。连续弹出 `A`、`B`、`C` 后，队列实际应为 `[]`；随堂笔记中的 `# ['A', 'B', 'C']` 是注释错误。`pop(0)` 每次都要移动后续元素，队列较长时开销为 `O(q)`，所以本实现的层序操作最坏可能达到 `O(n²)`；这里保留课程写法，不替换成其他队列类型。


## 十、二叉树的广度优先遍历

广度优先遍历（BFS）也叫层序遍历：先访问根，再逐层访问左、右孩子。遍历时把当前结点的非空孩子追加到队列尾部，下一次从队头取出。空树直接返回。对无权或等权边的图，BFS 按层扩展可以找到最短边数路径；这句话不能无条件推广到带权图。

在课件中方法名曾拼成 `breadh_travel`，随堂完整代码使用正确的 `breadth_travel`，本篇沿用后者。上面的 `BinaryTree` 类已经包含该方法。


### 层序插入和遍历的调用示例

```python
bt = BinaryTree()
for item in range(10):
    bt.add(item)

bt.breadth_travel()
print()
bt.preorder_travel(bt.root)
print()
bt.inorder_travel(bt.root)
print()
bt.postorder_travel(bt.root)
```

依次加入 `0` 到 `9` 时，课程示例的层序结果是：`0 1 2 3 4 5 6 7 8 9`。这里的数字只是讲义给出的预期示例，不是本 Notebook 的运行输出。


## 十一、二叉树的三种深度优先遍历

深度优先遍历（DFS）沿一条分支尽可能向下，再回溯处理其他分支。三种递归遍历只改变“访问根结点”的位置：

- **先序**：根 → 左 → 右；
- **中序**：左 → 根 → 右；
- **后序**：左 → 右 → 根。

递归遇到 `None` 就停止。随堂方法接收 `root` 参数，调用时要传入 `bt.root`。每个结点访问一次，时间复杂度为 `O(n)`；递归栈空间为 `O(h)`，`h` 是树高。

![0 到 9 的二叉树及遍历结果](07-img/06-二叉树遍历结果.jpg)

对依次加入 `0` 到 `9` 的课程示例，预期结果为：

- 层序：`0 1 2 3 4 5 6 7 8 9`
- 先序：`0 1 3 7 8 4 9 2 5 6`
- 中序：`7 3 8 1 9 4 0 5 2 6`
- 后序：`7 8 3 9 4 1 5 6 2 0`


## 十二、由遍历结果反推二叉树

当结点标识唯一时，**先序 + 中序**，或**后序 + 中序**，可以唯一确定一棵二叉树；只给先序和后序通常不能唯一确定。推导步骤如下：

1. 先序的第一个结点，或后序的最后一个结点，是根。
2. 在中序序列中找到根，根左侧属于左子树，右侧属于右子树。
3. 按相同规则分别切分左右子树的遍历序列，递归处理，直到每组只剩一个结点。

例如先序以 `0` 开头，中序中 `0` 把序列分成左侧 `7 3 8 1 9 4` 和右侧 `5 2 6`；左子树根是先序中的 `1`，右子树根是 `2`，继续划分即可。

![先序和中序反推二叉树](07-img/07-遍历反推二叉树.jpg)


## 十三、课程小结与易错点

1. 稳定性关注相同关键字记录的相对次序；冒泡和插入稳定，选择不稳定。
2. 冒泡、选择、插入都原地修改列表；选择排序即使已有序也要做 `O(n²)` 比较。
3. 二分查找必须建立在有序、可按下标访问的数据上；循环版避免切片复制。
4. 树是层次结构，二叉树的左右孩子有区别；B 树是多路查找树，不是普通二叉树。
5. 完全二叉树性质中的层数、编号和 Python 的 0 基下标要区分；深度公式要取整。
6. 本课的 `add` 按层序找空位，构造的是完全形状，不是 BST；层序遍历用 `pop(0)` 时要考虑移动开销。
7. 先序、中序、后序分别对应“根左右”“左根右”“左右根”，空子树是递归终止条件。
